# Create William T. Grant Foundation Grant Awards (GRANT PATTERN, Method 5 sitemap+HTML)

The William T. Grant Foundation is a US philanthropic funder supporting research on reducing inequality in youth outcomes and improving the use of research evidence in policy and practice, plus capacity-building grants to youth-serving organizations (largely in New York City). This ingest covers the foundation's **public grants archive** (`wtgrantfoundation.org/grants/...`), one row per grant.

**Method 5 (sitemap + static HTML).** The scraper (`scripts/local/wt_grant_to_s3.py`) enumerates every grant URL from the Yoast sitemaps (`/grants-sitemap.xml` + `/grants-sitemap2.xml`) and parses each grant detail page's "About This Grant" sidebar (`<div class="about-post-item">` rows). Plain `requests` + BeautifulSoup; no browser automation.

**Awarding body:** William T. Grant Foundation - F4320306360 (US, no ROR, DOI 10.13039/100001143)

**Source:** `wtgrantfoundation.org/grants/{slug}`. Each grant carries a lead (a **Principal Investigator** — named person at an institution — OR a **Grantee Organization** for org-level youth-service grants), an optional **Co-Principal Investigator**, **Grant Period**, **Programs**, optional **Focus Areas** / **Topics**, and (on ~78% of pages) a **Grant Amount** in USD.

**Schema choices:**
- One row per grant. `funder_award_id` = the source URL slug, e.g. `policing-and-protesting-juvenile-justice-inequality` (stable, unique, source-authoritative).
- `funding_type` = `grant` uniformly.
- `funder_scheme` = the grant's **Program** (Research Grants, William T. Grant Scholars, Mentoring Grant, Capacity-Building and Communications, Youth Service Improvement Grant, Youth Service Capacity-Building Grants, ...).
- **Amount IS published on ~78% of grants** (`$109,766.10` form) → `amount` (DOUBLE) + `currency` = `USD` where present, NULL otherwise. **§6.7 is NOT waived** — Sloan-style partial coverage: the figure is populated wherever the source posts one, never imputed. (A single grant posts `$0`; it is treated as NULL, not a real award amount.)
- `lead_investigator` = the named Principal Investigator (person) when present, else an org-only lead. `affiliation.name` = the grantee organization in both cases. `affiliation.country` is NULL — the source carries no city/region/country field, and country is never guessed (CLAUDE.md).
- `co_lead_investigator` = the named Co-Principal Investigator (person) when present (~25% of grants), with the same affiliation.
- `start_year` / `end_year` parsed from the Grant Period (`Month YYYY – Month YYYY`); `start_date` / `end_date` left NULL (the source gives month precision, not day, so no false-precision dates are asserted).
- `description` = the grant's OpenGraph description (abstract).

**S3 location:** `s3a://openalex-ingest/awards/wt_grant/wt_grant_grants.parquet`

**Provenance:** `wt_grant_foundation` (verified count=0 on production 2026-05-29).

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wt_grant_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wt_grant/wt_grant_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.wt_grant_raw;

## Step 1.5: Inspect raw + program/year/coverage

Expected: ~1,563 grants, 2004-2026 (grant periods extend into future end-years). title / program / grantee_org / start_year near-100%; amount ~78%; given_name ~64% (research/scholar grants have a named PI, youth-service grants are org-level).

In [ ]:
%sql
DESCRIBE openalex.awards.wt_grant_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.wt_grant_raw LIMIT 5;

In [ ]:
%sql
-- Per-(program, start_year) row counts + coverage.
SELECT program, start_year, COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       COUNT(given_name) AS has_pi,
       COUNT(grantee_org) AS has_org
FROM openalex.awards.wt_grant_raw
GROUP BY program, start_year
ORDER BY start_year DESC, rows DESC;

In [ ]:
%sql
-- Top grantee organizations (sanity check parsing didn't bleed fields).
SELECT grantee_org, COUNT(*) AS rows
FROM openalex.awards.wt_grant_raw
WHERE grantee_org IS NOT NULL
GROUP BY grantee_org
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast - verify William T. Grant Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306360;

## Step 2: Transform to award schema

Per-row `display_name` = `{program} - {grantee_org} ({start_year})`. `lead_investigator` is the named Principal Investigator (person) when present, else an org-only lead; `affiliation.name` = grantee org and `affiliation.country` is NULL (source carries no location field). `co_lead_investigator` is the named Co-Principal Investigator when present.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wt_grant_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306360  -- William T. Grant Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(COALESCE(n.program, 'Grant'), ' - ',
           COALESCE(n.grantee_org, NULLIF(CONCAT_WS(' ', n.given_name, n.family_name), ''), n.title),
           CASE WHEN n.start_year IS NOT NULL THEN CONCAT(' (', n.start_year, ')') ELSE '' END) AS display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN n.currency ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.program AS funder_scheme,
    'wt_grant_foundation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    TRY_CAST(n.end_year AS INT) AS end_year,
    CASE
        WHEN n.grantee_org IS NOT NULL OR n.given_name IS NOT NULL THEN struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.grantee_org AS name,
                CAST(NULL AS STRING) AS country,  -- source has no location field; never guessed
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CASE
        WHEN n.copi_given_name IS NOT NULL OR n.copi_family_name IS NOT NULL THEN struct(
            n.copi_given_name AS given_name,
            n.copi_family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                COALESCE(n.copi_org, n.grantee_org) AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.wt_grant_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 151

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wt_grant_foundation' AND priority = 151;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    151 AS priority  -- William T. Grant Foundation grants
FROM openalex.awards.wt_grant_awards;

## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived, partial): the foundation posts a dollar figure on ~78% of grant pages, so `pct_amount` should be ~78%. Other completeness checks: grantee_org / display_name near-100%, description high, start_year 100%.

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.wt_grant_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.wt_grant_awards;

In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_lead,
    COUNT(co_lead_investigator) AS has_colead,
    COUNT(start_year) AS has_start_year,
    COUNT(end_year) AS has_end_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.wt_grant_awards;

In [ ]:
%sql
-- §6.7 amount check (NOT waived, partial). Expect pct_amount ~78% and currency = [USD].
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.wt_grant_awards;

In [ ]:
%sql
-- Program (scheme) split.
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       COUNT(amount) AS has_amount,
       ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.wt_grant_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.wt_grant_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme AS program, funding_type, start_year, end_year, amount, currency,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       lead_investigator.affiliation.name AS org
FROM openalex.awards.wt_grant_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;